# Can we predict late deliveries?
Target: `late_delivery`, predicted at dispatch for future consignments, so the split is temporal.

- Each `consignment` appears once, so no entity grouping is needed; the test set is the **latest 20% of rows by `dispatched_on`**.
- `claim_paid_eur` is paid on a claim settled *after* delivery, so it is not known at dispatch. It is dropped (leakage), as is the identifier `consignment`.
- Features: `carrier`, `distance_km`, `weight_kg`, `parcels`, plus hour and day of week of dispatch.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, brier_score_loss
df=pd.read_csv('data.csv',parse_dates=['dispatched_on']).sort_values('dispatched_on').reset_index(drop=True)
assert df.consignment.is_unique
df['hour']=df.dispatched_on.dt.hour; df['dow']=df.dispatched_on.dt.dayofweek
print(df.shape, df.late_delivery.mean())

## 1. Hold out the test set (latest 20% of rows)

In [ ]:
n_test=int(len(df)*0.2)
train,test=df.iloc[:-n_test].copy(),df.iloc[-n_test:].copy()
print(train.dispatched_on.max(), test.dispatched_on.min())
FEATS=['carrier','distance_km','weight_kg','parcels','hour','dow']
CAT=['carrier']; NUM=[f for f in FEATS if f not in CAT]
Xtr,ytr=train[FEATS],train.late_delivery
print('train late rate',ytr.mean())

## 2. Candidate models (named before any scoring)
1. `baseline`: training base rate.
2. `logreg`: one-hot carrier + standardised numerics, L2 logistic regression (C=1).
3. `hgb_d{2,3}_lr{0.05,0.1}`: gradient boosting, 200 iterations, four fixed settings.

Selection metric: mean CV average precision (about 21% positives). Threshold: maximises F1 on pooled out-of-fold predictions of the chosen model.

In [ ]:
def prep(): return ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),CAT),('n',StandardScaler(),NUM)])
def hgb(d,lr):
    return make_pipeline(ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),CAT)],remainder='passthrough'),
                         HistGradientBoostingClassifier(max_depth=d,learning_rate=lr,max_iter=200,random_state=0))
CANDS={'baseline':lambda:DummyClassifier(strategy='prior'),
       'logreg':lambda:make_pipeline(prep(),LogisticRegression(C=1.0,max_iter=1000))}
for d in (2,3):
    for lr in (0.05,0.1): CANDS[f'hgb_d{d}_lr{lr}']=(lambda d=d,lr=lr:hgb(d,lr))
print(list(CANDS))

## 3. Time-ordered cross-validation inside the training rows (pipelines refit preprocessing per fold)

In [ ]:
tscv=TimeSeriesSplit(n_splits=5)
oof={}; rows=[]
for name,mk in CANDS.items():
    p=np.full(len(train),np.nan); aps=[];aucs=[]
    for tr,va in tscv.split(Xtr):
        m=mk().fit(Xtr.iloc[tr],ytr.iloc[tr]); pv=m.predict_proba(Xtr.iloc[va])[:,1]; p[va]=pv
        aps.append(average_precision_score(ytr.iloc[va],pv)); aucs.append(roc_auc_score(ytr.iloc[va],pv))
    oof[name]=p; rows.append((name,np.mean(aps),np.std(aps),np.mean(aucs)))
cv=pd.DataFrame(rows,columns=['model','cv_AP','cv_AP_sd','cv_AUC']).sort_values('cv_AP',ascending=False)
print(cv.round(3))

## 4. Choose model and threshold on CV only

In [ ]:
best=cv.iloc[0].model
mask=~np.isnan(oof[best])
ths=np.linspace(0.05,0.8,76)
f1s=[f1_score(ytr[mask],oof[best][mask]>=t) for t in ths]
thr=float(ths[int(np.argmax(f1s))])
print('chosen model:',best,'| threshold:',round(thr,3),'| CV F1:',round(max(f1s),3))

### Recorded choice
The model and threshold printed above are final, fixed before the test set is scored.

## 5. Refit on all training rows; score once on the test set

In [ ]:
final=CANDS[best]().fit(Xtr,ytr)
pt=final.predict_proba(test[FEATS])[:,1]; yt=test.late_delivery
base=CANDS['baseline']().fit(Xtr,ytr).predict_proba(test[FEATS])[:,1]
res=pd.DataFrame({'AUC':[roc_auc_score(yt,pt),roc_auc_score(yt,base)],
  'AP':[average_precision_score(yt,pt),average_precision_score(yt,base)],
  'Brier':[brier_score_loss(yt,pt),brier_score_loss(yt,base)]},index=[best,'baseline'])
res.loc[best,'F1@thr']=f1_score(yt,pt>=thr); res.loc[best,'precision@thr']=yt[pt>=thr].mean(); res.loc[best,'recall@thr']=(pt[yt==1]>=thr).mean()
print('test late rate',yt.mean()); print(res.round(3))

## 6. Conclusion
The headline is the test-set row above versus the base-rate baseline. `claim_paid_eur` was excluded as post-outcome information.